# Learned Stage-Wise Curvature Tuning on DermaMNIST

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the DermaMNIST dataset using an ImageNet-pretrained ResNet-18.

DermaMNIST is a medical image-classification dataset containing dermatological lesion images. Its visual domain differs from the natural images used for ImageNet pretraining, making it useful for testing whether Learned SW-CT remains effective under substantial domain shift.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare the computational cost of Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Thu Oct  1 23:59:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 575 (delta 153), reused 181 (delta 103), pack-reused 328 (from 1)
Receiving objects: 100% (575/575), 6.93 MiB | 5.91 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
068c466 (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) Updated aircraft with complete results


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 71.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 65.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 M

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## DermaMNIST Dataset

Before running the experiments, the DermanMNIST data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_medmnist/dermamnist",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md: 0.00B [00:00, ?B/s]

medmnist.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train samples: 7007
Validation samples: 1003
Test samples: 2005
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([4, 5, 2, 5, 5, 3, 5, 5, 5, 5])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, DermaMNIST dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 00:01:35.039 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs20.log
2026-10-02 00:01:35.041 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 173MB/s]
2026-10-02 00:01:38.166 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 00:01:38.214 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 00:01:38.214 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 00:01:50.606 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.679224, Accuracy: 37.50%
2026-10-02 00:01:50.884 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.297883, Accuracy: 62.56%
2026-10-02 00:01:51.168 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Los

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 00:20:57.365 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs20.log
2026-10-02 00:20:57.366 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 00:21:00.408 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 00:21:00.426 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 00:21:00.427 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 00:21:14.971 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.071713, Accuracy: 3.12%
2026-10-02 00:21:15.315 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.400004, Accuracy: 53.12%
2026-10-02 00:21:15.652 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.158240, Accuracy: 61.74%
2026-10-02 00:21:15.863 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.848847, Val Accuracy: 72.88%
2026-10-02 00:21:15.864 | INFO     | train:linear_probe:190 - New best valid

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 00:41:12.057 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs20.log
2026-10-02 00:41:12.059 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 00:41:15.058 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 00:41:15.077 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 00:41:15.077 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 00:41:31.947 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.764945, Accuracy: 3.12%
2026-10-02 00:41:32.298 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.716551, Accuracy: 38.95%
2026-10-02 00:41:32.653 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.327978, Accuracy: 54.06%
2026-10-02 00:41:32.896 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.846382, Val Accuracy: 71.29%
2026-10-02 00:41:32.897 | INFO     | train:linear_probe:190 - New best valid

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:01:39.964 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 01:01:40.029 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:01:43.077 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 01:01:43.077 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:01:43.119 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 01:01:43.119 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:01:43.120 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 01:01:44.461 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.196857, Accuracy: 6.25%
2026-10-02 01:02:00.060 | INFO     | train:train_

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:14:23.165 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 01:14:23.280 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:14:26.181 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 01:14:26.181 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:14:26.213 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 01:14:26.214 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:14:26.214 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 01:14:27.158 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.057349, Accuracy: 9.38%
2026-10-02 01:14:42.601 | INFO     | train:train_

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:27:01.155 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 01:27:01.266 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:27:04.179 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 01:27:04.179 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:27:04.212 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 01:27:04.213 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:27:04.213 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 01:27:05.267 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.000003, Accuracy: 21.88%
2026-10-02 01:27:20.694 | INFO     | train:train

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:39:36.910 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-02 01:39:36.971 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:39:39.839 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 01:39:39.840 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:39:39.871 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 01:39:39.872 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:39:39.872 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 01:39:40.820 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.205614, Accuracy: 6.25%
2026-10-02 01:39:56.364 | INFO     | 

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:52:11.172 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-02 01:52:11.277 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:52:14.176 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 01:52:14.176 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:52:14.217 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 01:52:14.218 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:52:14.218 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 01:52:15.183 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.026596, Accuracy: 6.25%
2026-10-02 01:52:30.687 | INFO     

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 02:04:46.798 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-02 02:04:46.852 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 02:04:49.827 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 02:04:49.827 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 02:04:49.859 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 02:04:49.860 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 02:04:49.860 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 02:04:50.830 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.975294, Accuracy: 18.75%
2026-10-02 02:05:06.312 | INFO    

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 02:17:20.015 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs30.log
2026-10-02 02:17:20.017 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 02:17:22.996 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 02:17:23.009 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 02:17:23.009 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 02:17:39.033 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.679224, Accuracy: 37.50%
2026-10-02 02:17:39.343 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.297883, Accuracy: 62.56%
2026-10-02 02:17:39.678 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.114770, Accuracy: 66.03%
2026-10-02 02:17:39.900 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.848266, Val Accuracy: 70.39%
2026-10-02 02:17:39.902 | INFO     | train:linear_probe:190 - New best vali

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 02:41:48.947 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs30.log
2026-10-02 02:41:48.950 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 02:41:51.926 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 02:41:51.941 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 02:41:51.941 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 02:42:06.345 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.071713, Accuracy: 3.12%
2026-10-02 02:42:06.653 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.400004, Accuracy: 53.12%
2026-10-02 02:42:06.969 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.158240, Accuracy: 61.74%
2026-10-02 02:42:07.171 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.848847, Val Accuracy: 72.88%
2026-10-02 02:42:07.172 | INFO     | train:linear_probe:190 - New best valid

In [14]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 03:06:23.206 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs30.log
2026-10-02 03:06:23.208 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 03:06:26.206 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 03:06:26.232 | INFO     | __main__:main:116 - Number of trainable parameters: 3591
2026-10-02 03:06:26.232 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 03:06:40.868 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.764945, Accuracy: 3.12%
2026-10-02 03:06:41.219 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.716551, Accuracy: 38.95%
2026-10-02 03:06:41.545 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.327978, Accuracy: 54.06%
2026-10-02 03:06:41.758 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.846382, Val Accuracy: 71.29%
2026-10-02 03:06:41.759 | INFO     | train:linear_probe:190 - New best valid

In [15]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 03:31:05.955 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 03:31:06.021 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 03:31:09.698 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 03:31:09.698 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 03:31:09.724 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 03:31:09.724 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 03:31:09.725 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 03:31:10.681 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.196857, Accuracy: 6.25%
2026-10-02 03:31:26.228 | INFO     | train:train_

In [16]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 03:49:52.593 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 03:49:52.670 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 03:49:55.673 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 03:49:55.673 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 03:49:55.710 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 03:49:55.710 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 03:49:55.711 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 03:49:56.683 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.057349, Accuracy: 9.38%
2026-10-02 03:50:12.141 | INFO     | train:train_

In [17]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 04:08:41.295 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 04:08:41.395 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 04:08:44.370 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 04:08:44.371 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 04:08:44.404 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 04:08:44.405 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 04:08:44.405 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 04:08:45.512 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.000003, Accuracy: 21.88%
2026-10-02 04:09:01.037 | INFO     | train:train

In [18]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 04:27:30.604 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-02 04:27:30.667 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 04:27:33.665 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 04:27:33.665 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 04:27:33.706 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 04:27:33.707 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 04:27:33.707 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 04:27:34.822 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.205614, Accuracy: 6.25%
2026-10-02 04:27:50.422 | INFO     | 

In [19]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 04:46:18.924 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-02 04:46:19.003 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 04:46:21.863 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 04:46:21.863 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 04:46:21.894 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 04:46:21.895 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 04:46:21.895 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 04:46:22.914 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.026596, Accuracy: 6.25%
2026-10-02 04:46:38.363 | INFO     

In [20]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 05:05:06.886 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-02 05:05:06.962 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 05:05:09.866 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 05:05:09.867 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 05:05:09.899 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 05:05:09.899 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 05:05:09.900 | INFO     | __main__:main:439 - Total trainable parameters: 3595
2026-10-02 05:05:10.927 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.975294, Accuracy: 18.75%
2026-10-02 05:05:26.384 | INFO    